In [0]:
%sql
use catalog py_spark_catalog;


In [0]:
%sql
select * from py_spark_catalog.big_mart_schema.big_mart_sales limit 10

spark.read.csv() → reads the CSV file.                                    
header=True → first row contains column names.                            
inferSchema=True → Spark tries to automatically determine data types.     
df → Spark DataFrame.                                                     
display(df) → displays the DataFrame in Databricks.


In [0]:
df = spark.read.csv(
    "/Volumes/py_spark_catalog/big_mart_schema/csv_volume/BigMart Sales.csv",
    header=True,
    inferSchema=True
)

display(df)

In [0]:
df.show()

In [0]:
df.count()

In [0]:
df.describe()

In [0]:
df.printSchema()
#This tells you the structure of your dataset.

###Define your own schema

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    DecimalType
)

#or
#from pyspark.sql.types import *
from pyspark.sql.functions import *
schema = StructType([
    StructField("Item_Identifier", StringType(), True),
    StructField("Item_Weight", DoubleType(), True),
    StructField("Item_Fat_Content", StringType(), True),
    StructField("Item_Visibility", DoubleType(), True),
    StructField("Item_Type", StringType(), True),
    StructField("Item_MRP",DecimalType(10, 2), True),
    StructField("Outlet_Identifier", StringType(), True),
    StructField("Outlet_Establishment_Year", IntegerType(), True),
    StructField("Outlet_Size", StringType(), True),
    StructField("Outlet_Location_Type", StringType(), True),
    StructField("Outlet_Type", StringType(), True),
    StructField("Item_Outlet_Sales", DecimalType(12, 2), True)
])

In [0]:
df = spark.read.csv(
    "/Volumes/py_spark_catalog/big_mart_schema/csv_volume/BigMart Sales.csv",
    header=True,
    schema=schema
)
df.printSchema()

In [0]:
df.columns
#gives the column names

In [0]:
df.dtypes
#gives the column names and their data types.

In [0]:
#df.select("Item_Identifier", "Item_Type", "Item_MRP").show()
df.select("Item_Identifier", "Item_Type", "Item_MRP").display()
#display(df.select(col('Item_Identifier'), col('Item_Type'), col('Item_MRP')))


In [0]:
df.select(
    "Item_Identifier",
    "Item_MRP",
    (df.Item_MRP * 2).alias("Double_MRP")
).show()

In [0]:
df.filter(df.Item_MRP > 200).show()
#or df.where(df.Item_MRP > 200).show()

In [0]:
df.filter(
    (df.Item_MRP > 200) & 
    (df.Item_Type == "Dairy")
).show()

In [0]:
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin("Tier 1", "Tier 2"))).display()

df.filter(
    (df.Outlet_Size.isNull()) &
    (df.Outlet_Location_Type.isin("Tier 1", "Tier 2"))
).display()


###column renamed 

In [0]:
df = df.withColumnRenamed("Item_Weight", "Item_Wt")


In [0]:
df.printSchema()

###add column

In [0]:
df = df.withColumn(
    "Item_MRP_Double",
    df.Item_MRP * 2
)

df = df.withColumn(
    "Total_Sales",
    df.Item_MRP * df.Item_Outlet_Sales
)

from pyspark.sql.functions import lit

df = df.withColumn(
    "Country",
    lit("India")
)

from pyspark.sql.functions import when

df = df.withColumn(
    "MRP_Category",
    when(df.Item_MRP >= 200, "High")
    .when(df.Item_MRP >= 100, "Medium")
    .otherwise("Low")
)

#If the column already exists → it replaces it.
# df = df.withColumn(
#     "Item_MRP",
#     df.Item_MRP * 2
# )

df = df.withColumn(
    "Item_Fat_Content",
    regexp_replace("Item_Fat_Content", "Low Fat", "LF")
)

df = df.withColumn(
    "Item_Fat_Content",
    regexp_replace("Item_Fat_Content", "Regular", "REG")
)

#do both replacements in one expression
# df = df.withColumn(
#     "Item_Fat_Content",
#     regexp_replace(
#         regexp_replace("Item_Fat_Content", "Low Fat", "LF"),
#         "Regular",
#         "REG"
#     )
# )

#when multiple column to be change then use when statement it is better 

# df = df.withColumn(
#     "Item_Fat_Content",
#     when(df.Item_Fat_Content == "Low Fat", "LF")
#     .when(df.Item_Fat_Content == "Regular", "REG")
#     .when(df.Item_Fat_Content == "low fat", "LF")
#     .otherwise(df.Item_Fat_Content)
# )

df.display()

###Type casting 

In [0]:
df = df.withColumn(
    "Total_Sales",
    df.Total_Sales.cast("decimal(10,2)")
)
df. printSchema()

###sort() vs orderBy() in PySpark

In [0]:
# df.orderBy(df.Item_MRP.desc()).display()
#df.sort("Item_MRP")

df.orderBy(
    df.Item_Type.asc(),
    df.Item_MRP.desc()
).display()

###Limit

In [0]:
df.limit(10).display()

###DROP



In [0]:
# df = df.drop("Item_Visibility")

# df = df.drop(
#     "Item_Visibility",
#     "Outlet_Size",
#     "Outlet_Location_Type"
# )

# #Drop using a list of columns

# columns_to_drop = [
#     "Item_Visibility",
#     "Outlet_Size",
#     "Outlet_Location_Type"
# ]
# df = df.drop(*columns_to_drop)

###dropDuplicates() in PySpark

In [0]:
before = df.count()

df = df.dropDuplicates(['item_MRP'])

after = df.count()

print("Before:", before)
print("After:", after)
print("Duplicates removed:", before - after)



In [0]:
#distinct() does not accept column names as arguments. It works on the entire row.
df.distinct().count()
df.distinct().display()
#If you want distinct Item_MRP values only:
df.select("Item_MRP").distinct().count()
df.select("Item_MRP").distinct().display()

###UNION

The columns must have the same number and compatible data types, and Spark matches them by position, not by column name.

In [0]:
df1 = spark.createDataFrame(
    [
        ("FDA15", "Dairy", 249.8),
        ("DRC01", "Soft Drinks", 48.2)
    ],
    ["Item_Identifier", "Item_Type", "Item_MRP"]
)

df2 = spark.createDataFrame(
    [
        ("FDN15", "Meat", 141.6),
        ("FDX07", "Fruits", 107.7)
    ],
    ["Item_Identifier", "Item_Type", "Item_MRP"]
)

In [0]:
result = df1.union(df2)

result.display()

###unionByName

unionByName() matches columns by their names

In [0]:
df3 = spark.createDataFrame(
    [
        ("FDA15", "Dairy", 249.8,"regular"),
        ("DRC01", "Soft Drinks", 48.2,"regular")
    ],
    ["Item_Identifier", "Item_Type", "Item_MRP","Outlet_Type"]
)

df4 = spark.createDataFrame(
    [
        ("Meat", 141.6,"FDN15"),
        ("Fruits", 107.7,"FDX07")
    ],
    ["Item_Type", "Item_MRP","Item_Identifier"]
)

In [0]:
df1.unionByName(df4).display()

allowMissingColumns=True

In [0]:
result = df3.unionByName(
    df4,
    allowMissingColumns=True
)
result.display()

###PySpark String Functions

In [0]:
df = df.withColumn(
    "Item_Type",
    upper(df.Item_Type)
)

display(df)

In [0]:
df = df.withColumn(
    "Item_Type",
    lower(df.Item_Type)
)
display(df.limit(10))

In [0]:
df = df.withColumn(
    "Item_Type",
    initcap(df.Item_Type)
)

df = df.withColumn(
    "Item_Type",
    trim(df.Item_Type)
)

df = df.withColumn(
    "Item_Type",
    ltrim(df.Item_Type)
)

df = df.withColumn(
    "Item_Type",
    rtrim(df.Item_Type)
)

display(df.limit(10))

In [0]:
df.select(
    "Item_Identifier",
    length(df.Item_Identifier).alias("ID_Length")
).display()

In [0]:
df = df.withColumn(
    "Product_Info",
    concat(
        df.Item_Identifier,
        df.Item_Type
    )
)

display(df.select('Product_Info').limit(5))

In [0]:
df = df.withColumn(
    "Product_Info",
    concat_ws(
        " <=> ",
        df.Item_Identifier,
        df.Item_Type
    )
)

display(df.select('Product_Info').limit(5))

In [0]:
display(df.select(
    "Item_Identifier",
    substring(df.Item_Identifier, 2, 3).alias("Item_Prefix")
).limit(5))

In [0]:
#regexp_replace()
"""
removing everything except letters:

df = df.withColumn(
    "Item_Type",
    regexp_replace(df.Item_Type, "[^a-zA-Z ]", "")
)

"""

In [0]:
df.filter(
    df.Item_Type.contains("Dairy")
).display()

In [0]:
df.filter(
    df.Item_Identifier.startswith("FD")
).display()

In [0]:
df.filter(
    df.Item_Identifier.endswith("15")
).display()

In [0]:
df.filter(
    df.Item_Fat_Content.isin("LF", "REG")
).display()

In [0]:
df.select(
    split(df.Item_Type, " ").alias("Item_Type_Array")
).display()

In [0]:
"""
A very important BigMart cleaning example

Suppose your raw Item_Fat_Content contains:

Low Fat
low fat
LF
Regular
regular
REG

You want standardized values:

LF
REG

You could use:

df = df.withColumn(
    "Item_Fat_Content",
    upper(trim(df.Item_Fat_Content))
)

Then:

df = df.withColumn(
    "Item_Fat_Content",
    regexp_replace(
        regexp_replace(
            df.Item_Fat_Content,
            "LOW FAT",
            "LF"
        ),
        "REGULAR",
        "REG"
    )
)
"""

###DATE 

In [0]:
df = df.withColumn(
    "Today",
    current_date()
)
df = df.withColumn(
    "Current_Timestamp",
    current_timestamp()
)

display(df.limit(5))

In [0]:
# """
# to_date() — String → Date
# df = df.withColumn(
#     "Order_Date",
#     to_date(df.Order_Date)
# )
# STRING
#    ↓
# to_date()
#    ↓
# DATE
# """

df = df.withColumn(
    "Today",
    to_date(col("Today"), "dd/MM/yyyy")
)
df.printSchema()

In [0]:
df = df.withColumn(
    "Today",
    date_format(col("Today"), "dd-MM-yyyy")
)
df.printSchema()
# to_date = make it a date
# date_format = make it formatted text

In [0]:
df_test = spark.createDataFrame(
    [
        ("23/08/2026",),
        ("15/01/2025",),
        ("10/12/2024",)
    ],
    ["Today"]
)

df_test = df_test.withColumn(
    "Today",
    to_date(col("Today"), "dd/MM/yyyy")
)

df_test = df_test.withColumn(
    "Order_Year",
    year(col("Today"))
)
df_test = df_test.withColumn(
    "Order_Month",
    month(col("Today"))
)
df_test = df_test.withColumn(
    "Order_day",
    day(col("Today"))
)
df_test = df_test.withColumn(
    "Day_Of_Week",
   dayofweek(col("Today"))
)
df_test = df_test.withColumn(
    "Day_Of_Year",
   dayofyear(col("Today"))
)
df_test = df_test.withColumn(
    "Week_Of_Year",
   weekofyear(col("Today"))
)
#date diff
df_test = df_test.withColumn(
    "Days_Since_2024",
   datediff(col("Today"), to_date(lit("01/01/2024"), "dd/MM/yyyy"))
)
#date add
df_test = df_test.withColumn(
    "after 7 day s",
   date_add(col("Today"), 7)
)
#date sub
df_test = df_test.withColumn(
    "before 7 days",
   date_sub(col("Today"), 7)
)
#add_months
df_test = df_test.withColumn(
    "aftre 7 months",
   add_months(col("Today"), 7)
)
#months_between
df_test = df_test.withColumn(
    "months difference",
    months_between(df_test.Today, to_date(lit("01/01/2024"), "dd/MM/yyyy"))
)
#last day
df_test = df_test.withColumn(
    "last day",
    last_day(col("Today"))
)
#month start
df_test = df_test.withColumn(
    "month start",
    trunc(col("Today"),"month")
)
#year start
df_test = df_test.withColumn(
    "year start",
    trunc(col("Today"),"year")
)
df_test.printSchema()
display(df_test)

###Handelling Nulls

In [0]:
data = [
    ("Debendra", 22, 50000.0),
    ("Rahul", None, 45000.0),
    (None, 25, None),
    ("Amit", None, 30000.0),
    (None, None, None)
]

columns = ["Name", "Age", "Salary"]

df_null = spark.createDataFrame(data, columns)

display(df_null)

#chesk nulls
df_null.filter(
    df_null.Age.isNull()
).display()
#is not null
df_null.filter(
    df_null.Age.isNotNull()
).display()

#count null
df_null.select(
    count(when(df_null.Name.isNull(), True)).alias("Name_Nulls"),
    count(when(df_null.Age.isNull(), True)).alias("Age_Nulls"),
    count(when(df_null.Salary.isNull(), True)).alias("Salary_Nulls")
).display()

#remove rows containing NULL
#df_clean = df_null.dropna()

#Drop rows based on a specific column
#df_clean = df_null.dropna(subset=["Age"])

#replace null values 
#df_clean = df_null.fillna({"Age": 0})

#Fill different columns with different values
df_clean = df_null.fillna({
    "Name": "Unknown",
    "Age": 0,
    "Salary": 0.0
})

#Fill numeric NULLs using average
avg_age = df_null.select(
    avg("Age")
).first()[0]

df_clean = df_null.fillna({
    "Age": avg_age
})

#fill using median
median_age = df_null.approxQuantile(
    "Age",
    [0.5],
    0.01
)[0]
#"Find approximately the value at the 50th percentile of Age, with relative error up to 1%."
df_clean = df_null.fillna({
    "Age": median_age
})

#coalesce() can replace NULL with another column/value.
df_null = df_null.withColumn(
    "Age_Clean",
    coalesce(df_null.Age, lit(18))
)
display(df_null)


display(df_clean)

###array_contains

In [0]:
data = [
    (1, ["Python", "SQL", "Spark"]),
    (2, ["Java", "SQL"]),
    (3, ["Python", "Pandas"]),
    (4, ["C++", "Java"])
]

df_array = spark.createDataFrame(
    data,
    ["id", "skills"]
)


display(df_array)


df_array.filter(
    array_contains(df_array.skills, "Python")
).display()

#Create a TRUE/FALSE column
df_array = df_array.withColumn(
    "Has_Python",
    array_contains(df_array.skills, "Python")
)

display(df_array)
#Using it with when()
df_array = df_array.withColumn(
    "Python_User",
    when(
        array_contains(df_array.skills, "Python"),
        "Yes"
    ).otherwise("No")
)

display(df_array)

###groupBy()

In [0]:
data = [
    ("Electronics", 1000),
    ("Electronics", 1500),
    ("Clothing", 500),
    ("Clothing", 700),
    ("Food", 300),
    ("Food", 400)
]

df_group = spark.createDataFrame(
    data,
    ["Category", "Sales"]
)

display(df_group)

df_group.groupBy("Category").count().display()

df_group.groupBy("Category").agg(
    sum("Sales").alias("Total_Sales")
).display()


df_group.groupBy("Category").agg(
    avg("Sales").alias("Average_Sales")
).display()


df_group.groupBy("Category").agg(
    count("*").alias("Number_of_Records"),
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales"),
    min("Sales").alias("Minimum_Sales"),
    max("Sales").alias("Maximum_Sales")
).display()

###Collect List and Collect Set

In [0]:
from pyspark.sql.functions import *
data = [
    ("A", "Python"),
    ("A", "SQL"),
    ("A", "Spark"),
    ("A", "Python"),
    ("B", "Java"),
    ("B", "SQL"),
    ("B", "Python")
]

df_list = spark.createDataFrame(
    data,
    ["Student", "Skill"]
)

display(df_list)

df_list.groupBy("Student").agg(
    collect_list("Skill").alias("Skills")
).display()
#Removes duplicates
df_list.groupBy("Student").agg(
    collect_set("Skill").alias("Skills")
).display()

###collect_list() + sort_array()

In [0]:
df_list.groupBy("Student").agg(
    sort_array(
        collect_list("Skill")
    ).alias("Skills")
).display()

In [0]:
df_list.groupBy("Skill").count().display()

df_list.groupBy("Skill").agg(collect_list("Student").alias("Students")).display()

###PIVOT

In [0]:
data = [
    ("A", "Python", 100),
    ("A", "SQL", 200),
    ("A", "Spark", 300),
    ("B", "Python", 150),
    ("B", "SQL", 250),
    ("B", "Spark", 350)
]

df_pivot = spark.createDataFrame(
    data,
    ["Student", "Skill", "Score"]
)

display(df_pivot)

#it require a agg function
df_pivot.groupBy("Student").pivot("Skill").sum("Score").display()

#it also support multiple agg functions
df_pivot.groupBy("Student").pivot("Skill").agg(
    sum("Score").alias("Total_Score"),
    avg("Score").alias("Average_Score")
).display()

###When Otherwise

In [0]:
data = [
    ("Debendra", 22, 85000),
    ("Rahul", 17, 30000),
    ("Amit", 25, 55000),
    ("Rohit", 35, 120000),
    ("Suman", 16, 20000)
]

df_when = spark.createDataFrame(
    data,
    ["Name", "Age", "Salary"]
)

display(df_when)

#basic
df_when = df_when.withColumn(
    "Category",
    when(df_when.Age >= 18, "Adult")
    .otherwise("Minor")
)

display(df_when)

In [0]:
#Multiple conditions
df_when = df_when.withColumn(
    "Age_Group",
    when(df_when.Age < 18, "Minor")
    .when(df_when.Age <= 30, "Young Adult")
    .when(df_when.Age <= 50, "Adult")
    .otherwise("Senior")
)

display(df_when)

#using &   or |
df_when = df_when.withColumn(
    "Eligible",
    when(
        (df_when.Age >= 18) &
        (df_when.Salary >= 50000),
        "Yes"
    ).otherwise("No")
)

display(df_when)

#using isin
df_when = df_when.withColumn(
    "Selected",
    when(
        df_when.Age.isin(17, 18, 22),
        "Selected"
    ).otherwise("Not Selected")
)

###JOINS

In [0]:
customers = [
    (1, "Debendra", "Odisha"),
    (2, "Rahul", "Delhi"),
    (3, "Amit", "Mumbai"),
    (4, "Rohit", "Bangalore")
]

orders = [
    (101, 1, 500),
    (102, 2, 700),
    (103, 2, 300),
    (104, 5, 900)
]

df_customers = spark.createDataFrame(
    customers,
    ["customer_id", "name", "city"]
)

df_orders = spark.createDataFrame(
    orders,
    ["order_id", "customer_id", "amount"]
)

df_customers.display()
df_orders.display()

inner join

In [0]:
df_inner = df_customers.join(
    df_orders,
    df_customers.customer_id == df_orders.customer_id,
    "inner"
)

df_inner.display()

left join

In [0]:
df_left = df_customers.join(
    df_orders,
    df_customers.customer_id == df_orders.customer_id,
    "left"
)

df_left.display()

right join

In [0]:
df_right = df_customers.join(
    df_orders,
    df_customers.customer_id == df_orders.customer_id,
    "right"
)

df_right.display()

full outer

In [0]:
df_full = df_customers.join(
    df_orders,
    df_customers.customer_id == df_orders.customer_id,
    "full"
)

df_full.display()

###left semi 
Give me rows from the left DataFrame that have a match in the right DataFrame.
It returns only columns from the left DataFrame.

In [0]:
df_semi = df_customers.join(
    df_orders,
    df_customers.customer_id == df_orders.customer_id,
    "left_semi"
)

df_semi.display()

###Left Anti Join
Give me rows from the left DataFrame that do NOT have a match in the right DataFrame.

In [0]:
df_anti = df_customers.join(
    df_orders,
    df_customers.customer_id == df_orders.customer_id,
    "left_anti"
)

df_anti.display()

###Cross Join 
A cross join creates every possible combination of rows.

In [0]:
df_cross = df_customers.crossJoin(df_orders)

df_cross.display()

In [0]:
result = df_customers.join(
    df_orders,
    "customer_id",
    "inner"
).select(
    "customer_id",
    "name",
    "city",
    "order_id",
    "amount"
)

result.display()

###Window functions
For each row, which other rows should I consider when calculating something?
The basic Window syntax
from pyspark.sql.window import Window                                     
                                                                          
window_spec = Window \                                                    
    .partitionBy("column") \                                              
    .orderBy("column")                                                    
                                                                          
then:                                                                     
                                                                          
df.withColumn(                                                            
    "new_column",                                                         
    window_function("column").over(window_spec)                           
)                                                                         

In [0]:
data = [
    ("A", "Debendra", 5000),
    ("A", "Rahul", 7000),
    ("A", "Mahendra", 7000),
    ("A", "Amit", 6000),
    ("B", "Rohit", 4000),
    ("B", "Suman", 8000),
    ("B", "Raj", 3000)
]

df_window = spark.createDataFrame(
    data,
    ["Department", "Employee", "Salary"]
)

display(df_window)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
window_spec = Window.partitionBy("Department")

# aggreate over department and display for each row but if in simple group by then number of reduced
df_window.withColumn(
    "Avg_Salary",
    avg("Salary").over(window_spec)
).display()

###row_number()
Suppose you want to number employees based on salary.

In [0]:
window_spec = Window \
    .partitionBy("Department") \
    .orderBy(df_window.Salary.desc())
# .orderBy("Salary")
df_window.withColumn(
    "Row_Number",
    row_number().over(window_spec)
).display()

###rank()
when two employees have the same salary.
1 is repeated and 2 is skipped

In [0]:
df_window.withColumn(
    "Rank",
    rank().over(window_spec)
).display()

###dense_rank()
Values:      7000  7000  5000

row_number:     1     2     3
rank:           1     1     3
dense_rank:     1     1     2

row_number()
→ always unique number

rank()
→ same value gets same rank + gaps

dense_rank()
→ same value gets same rank + no gaps

In [0]:
df_window.withColumn(
    "Dense_Rank",
    dense_rank().over(window_spec)
).display()

###lag()
lag() looks at a previous row.

In [0]:
window_spec = Window \
    .partitionBy("Department") \
    .orderBy("Salary")

df_window.withColumn(
    "Previous_Salary",
    lag("Salary", 1).over(window_spec)
).display()

###lead()

In [0]:
df_window.withColumn(
    "Next_Salary",
    lead("Salary", 1).over(window_spec)
).display()

###Calculate salary difference

In [0]:
df_window = df_window.withColumn(
    "Previous_Salary",
    lag("Salary").over(window_spec)
)

df_window = df_window.withColumn(
    "Salary_Difference",
    df_window.Salary - df_window.Previous_Salary
)

display(df_window)

In [0]:
window_spec = Window.partitionBy("Department")

df_window.withColumn(
    "Department_Total",
    sum("Salary").over(window_spec)
).display()

###Percentage of department salary

In [0]:
df_window = df_window.withColumn(
    "Department_Total",
    sum("Salary").over(window_spec)
)

df_window = df_window.withColumn(
    "Salary_Percentage",
    (df_window.Salary / df_window.Department_Total) * 100
)

display(df_window)

###Running total

Salary    Running_Total                                                   
-----------------------                                                   
5000      5000                                                            
6000      11000                                                           
7000      18000                                                           
Meaning:                                                                  
5000                                                                      
5000 + 6000 = 11000                                                       
5000 + 6000 + 7000 = 18000 

future total:Window.unboundedFollowing

In [0]:
window_spec = Window \
    .partitionBy("Department") \
    .orderBy("Salary") \
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )

df_window.withColumn(
    "Running_Total",
    sum("Salary").over(window_spec)
).display()

###User Defined Functions (UDFs)

In [0]:
data = [
    ("Debendra", 22, 85000),
    ("Rahul", 17, 30000),
    ("Amit", 25, 55000),
    ("Rohit", 35, 120000),
    ("Suman", 16, 20000)
]

df = spark.createDataFrame(
    data,
    ["Name", "Age", "Salary"]
)

display(df)

###Type 1 — Basic Python UDF

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
def employee_type(age, salary):

    if age < 18:
        return "Minor"

    elif salary >= 80000:
        return "High Earner"

    else:
        return "Normal"
    
employee_udf = udf(
    employee_type,
    StringType()
)

df = df.withColumn(
    "Employee_Type",
    employee_udf(
        df.Age,
        df.Salary
    )
)

display(df)   

###UDF using decorator
Instead of:                                                               
salary_udf = udf(                                                         
    salary_category,                                                      
    StringType()                                                          
)                                                                         
@udf(returnType=StringType())

In [0]:
@udf(returnType=StringType())
def salary_category_udf(salary):

    if salary < 40000:
        return "Low"

    elif salary <= 80000:
        return "Medium"

    else:
        return "High"
    

df = df.withColumn(
    "Salary_Category",
    salary_category_udf(df.Salary)
)

display(df) 


###Different Types : IntegerType() , DoubleType() , BooleanType() , ArrayType(StringType())

In [0]:
@udf(returnType = ArrayType(StringType()))
def employee_tags(age, salary):

    tags = []

    if age < 18:
        tags.append("Minor")
    
    if age >= 18:
        tags.append("Major")

    if salary >= 80000:
        tags.append("High_Earner")

    if salary < 40000:
        tags.append("Low_Earner")

    return tags


df = df.withColumn(
    "Tags",
    employee_tags(
        df.Age,
        df.Salary
    )
)

display(df)

###UDF returning Struct                                                   


In [0]:
return_schema = StructType([
    StructField("Category", StringType(), True),
    StructField("Tax", DoubleType(), True)
])
def salary_details(salary):

    if salary < 40000:
        category = "Low"
    elif salary <= 80000:
        category = "Medium"
    else:
        category = "High"

    tax = salary * 0.10

    return (category, tax)

salary_details_udf = udf(
    salary_details,
    return_schema
)

df = df.withColumn(
    "Salary_Details",
    salary_details_udf(df.Salary)
)

display(df)

###Data writing
df.write \                                                                
  .format("format") \                                                     
  .mode("mode") \                                                         
  .save("path")     

 df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/tmp/students_csv")
                                                                           format : csv , json , parquet , delta                                    
 mode: append , overwrite , ignore , error / errorifexists

In [0]:
#general
df.write \
    .format("FORMAT") \
    .mode("MODE") \
    .option("KEY", "VALUE") \
    .partitionBy("COLUMN") \
    .save("PATH")
    
# CSV
df.write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .save("/path")

# Parquet
df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save("/path")

# Delta
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/path")

# Delta table
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("catalog.schema.table")

# Append
df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("catalog.schema.table")

# Partition
df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("State") \
    .saveAsTable("catalog.schema.table")

# Repartition
df.repartition(4)

# Repartition by column
df.repartition("State")

# Reduce partitions
df.coalesce(2)

###Spark SQL

In [0]:
data = [
    (1, "Debendra", "Odisha", "Laptop", 60000),
    (2, "Rahul", "Odisha", "Mobile", 25000),
    (3, "Amit", "Bihar", "Laptop", 55000),
    (4, "Rohit", "Odisha", "Tablet", 30000),
    (5, "Suman", "Bihar", "Mobile", 20000),
    (6, "Ankit", "Jharkhand", "Laptop", 70000),
    (7, "Priya", "Odisha", "Mobile", 28000),
    (8, "Sneha", "Bihar", "Tablet", 35000)
]

df = spark.createDataFrame(
    data,
    ["Order_ID", "Customer", "State", "Product", "Amount"]
)

display(df)

###DataFrame vs Spark SQL

In [0]:
df.filter(df["State"] == "Odisha").display()

In [0]:
"""SELECT *
FROM sales
WHERE State = 'Odisha';
But SQL doesn't automatically know what sales means.

We need to create a temporary view."""
df.createOrReplaceTempView("sales")
spark.sql("""
    SELECT *
    FROM sales
""").display()